# Lesson 9 — Map Cleaning & Nav2 Tuning

*A raw SLAM map is rarely ready for navigation. Grey zones, wall gaps, and noisy pixels will cause Nav2 to fail. This lesson shows you how to clean the map and tune Nav2 for your environment.*

---

## 🎬 Watch first — then do

<div style="padding:56.25% 0 0 0;position:relative;">
  <iframe src="https://player.vimeo.com/video/YOUR_VIDEO_ID"
          style="position:absolute;top:0;left:0;width:100%;height:100%;"
          frameborder="0" allow="autoplay; fullscreen; picture-in-picture" allowfullscreen>
  </iframe>
</div>

---

## 🗺️ Map Anatomy — .pgm & .yaml

SLAM translates sensor sweeps into a raw 2D pixel grid where color dictates traversability:

![Map anatomy: pgm pixel values and yaml metadata](images/nav2_architecture.png)

- **Value 0 (white)** — free space, the robot can drive here
- **Value 100 (black)** — solid wall or obstacle
- **Value -1 (grey)** — unknown territory, never scanned

The `.yaml` file stores the map resolution (meters per pixel) and origin coordinates.

---

## 🚀 Launch Nav2

With your cleaned map, launch Nav2:

```bash
ros2 launch turtlebot3_navigation2 navigation2.launch.py \
  use_sim_time:=True \
  map:=$HOME/my_map.yaml
```

In RViz, use **2D Pose Estimate** to tell Nav2 where the robot currently is on the map. Then use **Nav2 Goal** to send a destination.

---

## 📦 Reusing Your Map

Once you have a clean map you are happy with, you can skip the SLAM step in future sessions and go straight to navigation. Load your saved map directly:

```bash
ros2 launch turtlebot3_navigation2 navigation2.launch.py \
  use_sim_time:=True \
  map:=$HOME/my_map.yaml
```

Nav2 will localize the robot on the existing map instead of building a new one. Use **2D Pose Estimate** in RViz to tell Nav2 where the robot currently is.

---

## 🧠 Global vs Local Planner

Nav2 uses two planners working at different frequencies and scopes:

![Global planner vs local planner](images/global_vs_local_planner.png)

- **Global Planner** (~1 Hz) — sees the entire map, finds the optimal path from start to goal
- **Local Planner** (~20 Hz) — sees only the 3-meter radius, executes the path while dodging dynamic obstacles

---

## 💰 Costmaps

Planners don't navigate on the raw map — they navigate on a **costmap**: a grid where each cell has a cost. The planner finds the path with the lowest total cost.

![Costmap layers: raw map → cost grid → calculated path](images/costmap_layers.png)

Obstacle cells have maximum cost. Free space has zero cost. The planner routes through the zero-cost valleys.

---

## ⚙️ Parameter Tuning — Inflation Radius

The most important Nav2 parameter is the **inflation radius** — how far the costmap inflates obstacles to keep the robot away from walls.

![Inflation radius: 0.8m blocked vs 0.25m successful](images/nav2_slide16.jpg)

- **High radius (0.8m)** — very safe but blocks narrow corridors
- **Low radius (0.25m)** — fits narrow spaces but risks collisions

The params file is in your cloned workspace:

```bash
~/turtlebot3_ws/src/turtlebot3/turtlebot3_navigation2/param/waffle.yaml
```

Edit it:

```bash
nano ~/turtlebot3_ws/src/turtlebot3/turtlebot3_navigation2/param/waffle.yaml
```

Find and update:

```yaml
inflation_layer:
  inflation_radius: 0.25
```

Since you used `--symlink-install` when building, changes take effect immediately — no rebuild needed. Just relaunch Nav2.

> Rule of thumb: set inflation radius to at least half the robot's width. Turtlebot3 waffle is 0.28m wide → minimum `0.15m`, safe value `0.20-0.25m`.

## 📍 Waypoint Following

Complex environments with U-shaped corridors can trap the local planner. Use **waypoints** to break the goal into line-of-sight segments:

![Waypoint following: single goal failure vs waypoints success](images/pathfinding_A_star.png)

In RViz: click **Waypoint Mode** then place multiple waypoints along the route.

---

## 🔄 The Recovery Server

When the robot gets stuck, the Recovery Server automatically tries pre-programmed maneuvers:



1. **Spin 90°** — break free from a stuck state
2. **Backup** — reverse away from an obstacle
3. **Clear costmaps** — reset sensor-inflated costs
4. **Navigation Aborted** — if all recovery fails

---

## 🧠 Challenge

1. Your robot gets to the door of a room but refuses to enter — the path is blocked. The corridor is 0.6m wide. What inflation radius would you try?
2. Nav2 plans a path but the robot keeps oscillating in place. Which server is likely failing — Planner or Controller?
3. A grey zone in your map is between the robot and its goal. What happens when Nav2 tries to plan through it?

*Write your answers here:*

**1.**

**2.**

**3.**